In [3]:
pip install pyyaml


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import yaml
import os

folder_path = r"E:\odis"
all_balls = []
inning_labels = ['1st innings', '2nd innings', '3rd innings', '4th innings']

files = [f for f in os.listdir(folder_path) if f.endswith('.yaml')]
total = len(files)

for i, filename in enumerate(files):
    if i % 500 == 0:
        print(f"Processing {i}/{total} files...")

    filepath = os.path.join(folder_path, filename)
    with open(filepath, encoding='utf-8') as f:
        match = yaml.safe_load(f)

    info = match.get('info', {})
    match_id = filename.replace('.yaml', '')
    date = info.get('dates', [''])[0]
    venue = info.get('venue', '')
    city = info.get('city', '')
    teams = info.get('teams', ['', ''])
    season = info.get('season', '')
    gender = info.get('gender', '')
    match_type = info.get('match_type', '')
    toss_winner = info.get('toss', {}).get('winner', '')
    toss_decision = info.get('toss', {}).get('decision', '')
    outcome_winner = info.get('outcome', {}).get('winner', '')

    innings_data = match.get('innings', [])

    for inning_num, inning_label in enumerate(inning_labels):
        inning = None
        for item in innings_data:
            if inning_label in item:
                inning = item[inning_label]
                break
        if inning is None:
            continue

        batting_team = inning.get('team', '')
        bowling_team = teams[1] if batting_team == teams[0] else teams[0]
        target_runs = inning.get('target', {}).get('runs', None) if isinstance(inning.get('target'), dict) else None

        for delivery_dict in inning.get('deliveries', []):
            for ball_key, delivery in delivery_dict.items():
                parts = str(ball_key).split('.')
                over_num = int(parts[0])
                ball_num = int(parts[1]) if len(parts) > 1 else 0

                extras = delivery.get('extras', {}) or {}

                # Handle both dict and list formats for wicket
                raw_wicket = delivery.get('wicket', None)
                if isinstance(raw_wicket, dict):
                    wicket = raw_wicket
                    fielders = wicket.get('fielders', [])
                    # fielders can be list of strings or list of dicts
                    if fielders and isinstance(fielders[0], dict):
                        fielder_name = fielders[0].get('name', '')
                    else:
                        fielder_name = fielders[0] if fielders else ''
                    dismissal_kind = wicket.get('kind', '')
                    player_out = wicket.get('player_out', '')
                    is_wicket = 1
                elif isinstance(raw_wicket, list) and raw_wicket:
                    wicket = raw_wicket[0]
                    fielders = wicket.get('fielders', [])
                    if fielders and isinstance(fielders[0], dict):
                        fielder_name = fielders[0].get('name', '')
                    else:
                        fielder_name = fielders[0] if fielders else ''
                    dismissal_kind = wicket.get('kind', '')
                    player_out = wicket.get('player_out', '')
                    is_wicket = 1
                else:
                    fielder_name = ''
                    dismissal_kind = ''
                    player_out = ''
                    is_wicket = 0

                row = {
                    'match_id'       : match_id,
                    'date'           : date,
                    'season'         : season,
                    'venue'          : venue,
                    'city'           : city,
                    'gender'         : gender,
                    'match_type'     : match_type,
                    'team1'          : teams[0],
                    'team2'          : teams[1],
                    'toss_winner'    : toss_winner,
                    'toss_decision'  : toss_decision,
                    'outcome_winner' : outcome_winner,
                    'inning'         : inning_num + 1,
                    'batting_team'   : batting_team,
                    'bowling_team'   : bowling_team,
                    'target_runs'    : target_runs,
                    'over'           : over_num,
                    'ball'           : ball_num,
                    'batter'         : delivery.get('batsman', ''),
                    'non_striker'    : delivery.get('non_striker', ''),
                    'bowler'         : delivery.get('bowler', ''),
                    'runs_batter'    : delivery.get('runs', {}).get('batsman', 0),
                    'runs_extras'    : delivery.get('runs', {}).get('extras', 0),
                    'runs_total'     : delivery.get('runs', {}).get('total', 0),
                    'wides'          : extras.get('wides', 0),
                    'noballs'        : extras.get('noballs', 0),
                    'legbyes'        : extras.get('legbyes', 0),
                    'byes'           : extras.get('byes', 0),
                    'wicket'         : is_wicket,
                    'dismissal_kind' : dismissal_kind,
                    'player_out'     : player_out,
                    'fielder'        : fielder_name,
                    'fielder_sub'    : False,
                }
                all_balls.append(row)

df = pd.DataFrame(all_balls)
df.to_csv(r"E:\odi_international.csv", index=False)
print(f"Done! Total deliveries: {len(df)}")
print(df.head())

Processing 0/3136 files...
Processing 500/3136 files...
Processing 1000/3136 files...
Processing 1500/3136 files...
Processing 2000/3136 files...
Processing 2500/3136 files...
Processing 3000/3136 files...
Done! Total deliveries: 1659349
  match_id        date season                                   venue  \
0  1000887  2017-01-13         Brisbane Cricket Ground, Woolloongabba   
1  1000887  2017-01-13         Brisbane Cricket Ground, Woolloongabba   
2  1000887  2017-01-13         Brisbane Cricket Ground, Woolloongabba   
3  1000887  2017-01-13         Brisbane Cricket Ground, Woolloongabba   
4  1000887  2017-01-13         Brisbane Cricket Ground, Woolloongabba   

       city gender match_type      team1     team2 toss_winner  ...  \
0  Brisbane   male        ODI  Australia  Pakistan   Australia  ...   
1  Brisbane   male        ODI  Australia  Pakistan   Australia  ...   
2  Brisbane   male        ODI  Australia  Pakistan   Australia  ...   
3  Brisbane   male        ODI  Australi

In [7]:
import pandas as pd

df = pd.read_csv(r"E:\odi_international.csv", low_memory=False)

# Confirm Australia exists
aus_teams = [t for t in df['batting_team'].unique() if 'ustr' in t]
print("Teams:", aus_teams)

# Check our batters
aus_batters = ['MR Marsh', 'M Labuschagne', 'JP Inglis', 'C Green', 
               'L Scott', 'TM Head', 'AT Carey', 'MT Renshaw', 'C Connolly']

aus_df = df[df['batter'].isin(aus_batters) & (df['batting_team'] == 'Australia')].copy()

print(f"\nTotal deliveries to AUS batters: {len(aus_df)}")
print("\nDeliveries per batter:")
print(aus_df['batter'].value_counts())

Teams: ['Australia']

Total deliveries to AUS batters: 13176

Deliveries per batter:
batter
MR Marsh         3314
TM Head          2860
AT Carey         2560
M Labuschagne    2263
C Green           985
JP Inglis         833
MT Renshaw        223
C Connolly        138
Name: count, dtype: int64
